In [17]:
import numpy as np
import pandas as pd
from spreg import GM_Error
import libpysal
from libpysal.weights import Queen
import matplotlib.pyplot as plt
import geopandas as gpd
import scipy as sp
import arviz as az

RANDOM_SEED = 5781

# Transportation

## Data

In [18]:
tran_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_ONR_epa_climate.parquet'
tran_df = gpd.read_parquet(tran_par_file)
tran_wgt_file = '/work/hawkins_lab/vulcan/results/tran_weights.csv'
tran_wgt_df = pd.read_csv(tran_wgt_file)
# Create columns that assign decile numbers for each treatment variable
tran_df['treat_density'] = tran_df['d1a'] /1000
tran_df['treat_diversity'] = tran_df['d2b_e5mixa']
tran_df['treat_design'] = tran_df['d3a']
tran_df['treat_distance'] = tran_df['d4a'].replace(-99999,1500)  /1000
tran_df['treat_destination'] = tran_df['d5ar']  /1000

tran_df = pd.concat([tran_df,tran_wgt_df], axis=1)
tran_df = tran_df[(tran_df["value_weig"]>0) & (tran_df["totpop"]>0)]
tran_df["d1aa_cbsa"] = tran_df["totpop_cbsa"]/tran_df["ac_unpr_cbsa"] # true cbsa density

In [19]:
tran_df.groupby('cbsa').size()

cbsa
1.11          7
1.13         19
1.19         19
1.23         15
1.25         24
           ... 
49660.00    520
49700.00    111
49740.00    140
49780.00     75
49820.00      9
Length: 2013, dtype: int64

In [20]:
for c in ["treat_density","treat_diversity","treat_design","treat_distance","treat_destination", 
          "totpop_cbsa", "d1aa_cbsa","d2b_e5mix_cbsa","d3a_cbsa", "d4a_cbsa", "d5ar_cbsa"]:
    mu, sd = tran_df[c].mean(), tran_df[c].std()
    tran_df[c+"_z"] = (tran_df[c] - mu) / (sd if sd!=0 else 1.0)

for g in ["dens","div","des","dist","dest"]:
    mu, sd = tran_df[g].mean(), tran_df[g].std()
    tran_df["gps_"+g+"_z"] = (tran_df[g] - mu) / (sd if sd!=0 else 1.0)

tran_df["treat_density_ps"]   = tran_df["treat_density_z"]   * tran_df["gps_dens_z"]
tran_df["treat_diversity_ps"]  = tran_df["treat_diversity_z"] * tran_df["gps_div_z"]
tran_df["treat_design_ps"]     = tran_df["treat_design_z"]    * tran_df["gps_des_z"]
tran_df["treat_distance_ps"]  = tran_df["treat_distance_z"]  * tran_df["gps_dist_z"]
tran_df["treat_destination_ps"]= tran_df["treat_destination_z"] * tran_df["gps_dest_z"]

predictors = ["totpop_cbsa_z",
              "d1aa_cbsa_z",
              # "d1a_cbsa",
              "d2b_e5mix_cbsa_z",
              "d3a_cbsa_z",
              "d4a_cbsa_z", 
              "d5ar_cbsa_z",
              # "vmt_per_wo", # Add this one as the last of 3 transport-specific control variables from EPA data # Statistically insign
              "gps_dens_z",
              "gps_div_z",
              "gps_des_z",
              "gps_dist_z",
              "gps_dest_z",
              "treat_density_z",
              "treat_diversity_z",
              "treat_design_z",
              "treat_distance_z",
              "treat_destination_z",
              "treat_density_ps",
              "treat_diversity_ps",
              "treat_design_ps",
              "treat_distance_ps",
              "treat_destination_ps"]

X = tran_df.copy()

y = np.log(tran_df["value_weig"].replace(0, 10**-6) / tran_df["totpop"].replace(0, np.nan))

# X.loc[:,log_predictors] = np.log(X.loc[:,log_predictors].replace(0, 10**-6) ) # replace 0 with 10**-6

X1 = X.loc[:, predictors]
X_fe = pd.get_dummies(tran_df["cbsa_eig"], prefix="metro", drop_first=True).astype(int)
# Combine with covariates
X1_fe = pd.concat([X1, X_fe], axis=1)

In [21]:
X1_fe.describe()

,totpop_cbsa_z,d1aa_cbsa_z,d2b_e5mix_cbsa_z,d3a_cbsa_z,d4a_cbsa_z,d5ar_cbsa_z,gps_dens_z,gps_div_z,gps_des_z,gps_dist_z,...,metro_48660.0,metro_48700.0,metro_48900.0,metro_49180.0,metro_49340.0,metro_49420.0,metro_49620.0,metro_49660.0,metro_49700.0,metro_49740.0
count,2.153090e+05,2.153090e+05,2.153090e+05,2.153090e+05,2.153090e+05,2.153090e+05,2.153090e+05,2.153090e+05,2.153090e+05,2.153090e+05,...,215309.000000,215309.000000,215309.000000,215309.000000,215309.000000,215309.000000,215309.000000,215309.000000,215309.000000,215309.000000
mean,5.280171e-18,-1.709455e-17,5.174568e-16,1.161638e-16,2.956896e-17,-1.478448e-17,5.037283e-16,-2.310075e-17,-7.833794e-16,-5.354094e-16,...,0.000581,0.000520,0.000692,0.002020,0.003010,0.000692,0.001496,0.002415,0.000516,0.000650
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,...,0.024088,0.022802,0.026297,0.044903,0.054778,0.026297,0.038643,0.049085,0.022700,0.025491
min,-1.435677e-01,-7.598356e-02,-4.038903e+00,-1.655998e+00,-3.014747e-01,-2.209836e-01,-1.705869e+00,-4.358186e+00,-4.066107e+00,-1.858819e+00,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-1.381068e-01,-7.304580e-02,-5.273126e-01,-5.066844e-01,-3.014747e-01,-1.936033e-01,-7.441708e-01,-3.791984e-01,-3.833443e-01,-5.906956e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-1.314594e-01,-7.090506e-02,-1.110362e-01,-1.601592e-01,-3.014747e-01,-1.746438e-01,6.930880e-02,-6.521559e-02,-8.797784e-02,-2.814331e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,-1.150192e-01,-6.141998e-02,5.803649e-01,2.119698e-01,-3.014747e-01,-1.389854e-01,6.862973e-01,2.433275e-01,2.066323e-01,2.878807e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.027657e+01,1.831365e+01,3.499837e+00,1.369063e+01,8.817297e+00,9.957083e+00,1.880676e+00,2.186977e+01,3.817393e+01,1.916435e+01,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## Model

In [22]:
W = Queen.from_dataframe(tran_df)
W.transform = 'r'

model = GM_Error(y, X1_fe, w=W, name_y='y', name_x=list(X1_fe.columns))

/tmp/ipykernel_2650817/169258217.py:1: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  W = Queen.from_dataframe(tran_df)
/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 27 disconnected components.
 There are 12 islands with ids: 18616, 29427, 32710, 84610, 89731, 94628, 126088, 134362, 137261, 150235, 172351, 207415.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 18616, ' is an island (no neighbors)')
('WARNING: ', 29427, ' is an island (no neighbors)')
('WARNING: ', 32710, ' is an island (no neighbors)')
('WARNING: ', 84610, ' is an island (no neighbors)')
('WARNING: ', 89731, ' is an island (no neighbors)')
('WARNING: ', 94628, ' is an island (no neighbors)')
('WARNING: ', 126088, ' is an island (no neighbors)')
('WARNING: ', 134362, ' is an island (no neighbors)')
('WARNING: ', 137261, ' is an island (no neighbors)')
('WARNING: ', 150235, ' is an island (no neighbors)')
('WARNING: ', 172351, ' is an island (no neighbors)')
('WARNING: ', 207415, ' is an island (no neighbors)')


In [23]:
print(model.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GM SPATIALLY WEIGHTED LEAST SQUARES
------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :           y                Number of Observations:      215309
Mean dependent var  :      0.9804                Number of Variables   :         356
S.D. dependent var  :      1.1514                Degrees of Freedom    :      214953
Pseudo R-squared    :      0.2603

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT         1.25193         0.06030        20.76115         0.00000
       totpop_cbsa_z        -0.02918         0.05748        -0.50761         0.61172
         d1aa_cbsa_z         0.28795         0.35182         0.81847

## Model diagnostics

In [24]:
from esda.moran import Moran

residuals = model.y - model.predy

mi = Moran(residuals, W)
print(mi.I, mi.p_sim)

0.3851729660913437 0.001


# Residential Energy

## Data

In [25]:
res_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_RES_epa_climate.parquet'
res_df = gpd.read_parquet(res_par_file)
res_wgt_file = '/work/hawkins_lab/vulcan/results/res_weights.csv'
res_wgt_df = pd.read_csv(res_wgt_file)
# Create columns that assign decile numbers for each treatment variable
res_df['treat_density'] = res_df['d1a']
res_df['treat_diversity'] = res_df['d2b_e5mixa']

res_df = pd.concat([res_df,res_wgt_df], axis=1)
res_df = res_df[(res_df["value_weig"]>0) & (res_df["totpop"]>0)]
res_df["d1aa_cbsa"] = res_df["totpop_cbsa"]/res_df["ac_unpr_cbsa"] # true cbsa density

for c in ["treat_density","treat_diversity", "totpop_cbsa", "d1aa_cbsa","d2b_e5mix_cbsa"]:
    mu, sd = res_df[c].mean(), res_df[c].std()
    res_df[c+"_z"] = (res_df[c] - mu) / (sd if sd!=0 else 1.0)

for g in ["dens","div"]:
    mu, sd = res_df[g].mean(), res_df[g].std()
    res_df["gps_"+g+"_z"] = (res_df[g] - mu) / (sd if sd!=0 else 1.0)

res_df["treat_density_ps"]   = res_df["treat_density_z"]   * res_df["gps_dens_z"]
res_df["treat_diversity_ps"]  = res_df["treat_diversity_z"] * res_df["gps_div_z"]

predictors = ["totpop_cbsa_z",
              "d1aa_cbsa_z",
              # "d1a_cbsa",
              "d2b_e5mix_cbsa_z",
              # "vmt_per_wo", # Add this one as the last of 3 transport-specific control variables from EPA data # Statistically insign
              "gps_dens_z",
              "gps_div_z",
              "treat_density_z",
              "treat_diversity_z",
              "treat_density_ps",
              "treat_diversity_ps"]

X = res_df.copy()

y = np.log(res_df["value_weig"].replace(0, 10**-6) / res_df["totpop"].replace(0, np.nan)) # make per capita

X1 = X.loc[:, predictors]
X_fe = pd.get_dummies(res_df["cbsa_eig"], prefix="metro", drop_first=True).astype(int)
# Combine with covariates
X1_fe = pd.concat([X1, X_fe], axis=1)

## Model

In [26]:
W = Queen.from_dataframe(res_df)
W.transform = 'r'

model = GM_Error(y, X1_fe, w=W, name_y='y', name_x=list(X1_fe.columns))

/tmp/ipykernel_2650817/3261521901.py:1: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  W = Queen.from_dataframe(res_df)
/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 26 disconnected components.
 There are 11 islands with ids: 18612, 32704, 84580, 89701, 94598, 96633, 134310, 137205, 150177, 172293, 207355.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 18612, ' is an island (no neighbors)')
('WARNING: ', 32704, ' is an island (no neighbors)')
('WARNING: ', 84580, ' is an island (no neighbors)')
('WARNING: ', 89701, ' is an island (no neighbors)')
('WARNING: ', 94598, ' is an island (no neighbors)')
('WARNING: ', 96633, ' is an island (no neighbors)')
('WARNING: ', 134310, ' is an island (no neighbors)')
('WARNING: ', 137205, ' is an island (no neighbors)')
('WARNING: ', 150177, ' is an island (no neighbors)')
('WARNING: ', 172293, ' is an island (no neighbors)')
('WARNING: ', 207355, ' is an island (no neighbors)')


In [27]:
print(model.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GM SPATIALLY WEIGHTED LEAST SQUARES
------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :           y                Number of Observations:      215249
Mean dependent var  :     -0.1919                Number of Variables   :         344
S.D. dependent var  :      1.2203                Degrees of Freedom    :      214905
Pseudo R-squared    :      0.6369

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT        -1.08573         0.03443       -31.53501         0.00000
       totpop_cbsa_z         0.02060         0.01911         1.07845         0.28083
         d1aa_cbsa_z         0.05588         0.16237         0.34414

## Model diagnostics

In [28]:
residuals = model.y - model.predy
mi = Moran(residuals, W)
print(mi.I, mi.p_sim)

0.26569936977365693 0.001


## Residential Electricity

## Data

In [29]:
elec_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_SC2_epa_climate.parquet'
elec_df = gpd.read_parquet(elec_par_file)
elec_wgt_file = '/work/hawkins_lab/vulcan/results/elec_weights.csv'
elec_wgt_df = pd.read_csv(elec_wgt_file)
# Create columns that assign decile numbers for each treatment variable
elec_df['treat_density'] = elec_df['d1a']
elec_df['treat_diversity'] = elec_df['d2b_e5mixa']

elec_df = pd.concat([elec_df,elec_wgt_df], axis=1)
elec_df = elec_df[(elec_df["res_tc"]>0) & (elec_df["totpop"]>0)]
elec_df["d1aa_cbsa"] = elec_df["totpop_cbsa"]/elec_df["ac_unpr_cbsa"] # true cbsa density

for c in ["treat_density","treat_diversity", "totpop_cbsa", "d1aa_cbsa","d2b_e5mix_cbsa"]:
    mu, sd = elec_df[c].mean(), elec_df[c].std()
    elec_df[c+"_z"] = (elec_df[c] - mu) / (sd if sd!=0 else 1.0)

for g in ["dens","div"]:
    mu, sd = elec_df[g].mean(), elec_df[g].std()
    elec_df["gps_"+g+"_z"] = (elec_df[g] - mu) / (sd if sd!=0 else 1.0)

elec_df["treat_density_ps"]   = elec_df["treat_density_z"]   * elec_df["gps_dens_z"]
elec_df["treat_diversity_ps"]  = elec_df["treat_diversity_z"] * elec_df["gps_div_z"]

predictors = ["totpop_cbsa_z",
              "d1aa_cbsa_z",
              # "d1a_cbsa",
              "d2b_e5mix_cbsa_z",
              # "vmt_per_wo", # Add this one as the last of 3 transport-specific control variables from EPA data # Statistically insign
              "gps_dens_z",
              "gps_div_z",
              "treat_density_z",
              "treat_diversity_z",
              "treat_density_ps",
              "treat_diversity_ps"]

X = elec_df.copy()

y = np.log(elec_df["res_tc"].replace(0, 10**-6) / elec_df["totpop"].replace(0, np.nan)) # make per capita

X1 = X.loc[:, predictors]
X_fe = pd.get_dummies(elec_df["cbsa_eig"], prefix="metro", drop_first=True).astype(int)
# Combine with covariates
X1_fe = pd.concat([X1, X_fe], axis=1)

## Model

In [30]:
W = Queen.from_dataframe(elec_df)
W.transform = 'r'

model = GM_Error(y, X1_fe, w=W, name_y='y', name_x=list(X1_fe.columns))

/tmp/ipykernel_2650817/1655107388.py:1: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  W = Queen.from_dataframe(elec_df)
/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 34 disconnected components.
 There are 11 islands with ids: 3883, 85955, 91075, 95971, 98005, 127428, 135701, 138599, 140953, 151571, 173687.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 3883, ' is an island (no neighbors)')
('WARNING: ', 85955, ' is an island (no neighbors)')
('WARNING: ', 91075, ' is an island (no neighbors)')
('WARNING: ', 95971, ' is an island (no neighbors)')
('WARNING: ', 98005, ' is an island (no neighbors)')
('WARNING: ', 127428, ' is an island (no neighbors)')
('WARNING: ', 135701, ' is an island (no neighbors)')
('WARNING: ', 138599, ' is an island (no neighbors)')
('WARNING: ', 140953, ' is an island (no neighbors)')
('WARNING: ', 151571, ' is an island (no neighbors)')
('WARNING: ', 173687, ' is an island (no neighbors)')


In [31]:
print(model.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GM SPATIALLY WEIGHTED LEAST SQUARES
------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :           y                Number of Observations:      216623
Mean dependent var  :     -0.9630                Number of Variables   :         349
S.D. dependent var  :      0.8472                Degrees of Freedom    :      216274
Pseudo R-squared    :      0.6432

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT        -0.22716         0.02663        -8.52882         0.00000
       totpop_cbsa_z        -0.02007         0.01372        -1.46279         0.14353
         d1aa_cbsa_z        -0.01142         0.11597        -0.09850

## Model diagnostics

In [32]:
residuals = model.y - model.predy
mi = Moran(residuals, W)
print(mi.I, mi.p_sim)

0.4505847512445813 0.001
